# 02 — Data exploration

**Goal.** Understand the five raw files before building the player-season table.
Each section answers a question that drives a design choice of the project:

| File | Question | Decision it drives |
|---|---|---|
| `appearances` | Which period, which competitions, any gaps? | Usable seasons, how minutes are counted |
| `player_valuations` | How often are values updated? How many breakouts? | Definition of the target and its thresholds |
| `players` | Are birth date and position available? | Age feature, keep attackers and midfielders |
| `clubs` | What club information exists? | Measure of club strength (and leakage risks) |
| `competitions` | Which are leagues, which are cups? | League vs. cup minutes, league level |
| `Game_Lineup` | Important to know the details, take in acount the time really played by the player |
| Cross-check | How many player-seasons have both minutes and a value? | Real size of the sample |

Nothing in this notebook is used by the models directly: it only informs choices made in `src/`.

## 0. Setup

Import pandas and the project paths from `src/config.py`, so that no absolute path is written in the code.

In [58]:
import sys
sys.path.append("..")                        # allows importing from src/
from src import config
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column when printing a table

## 1. `appearances`  match-level performance

One row per player per match, with minutes, goals, assists and cards.

**What we expect to see:** about 1.9 million rows, matches from July 2012 to June 2026,
around 29,500 players and 48 competitions, and almost no missing values.

In [59]:
# Load the file and look at its size and columns
app = pd.read_csv(config.DATA_RAW / "appearances.csv")
print(app.shape)
print(app.columns.tolist())
app.head()

(1894350, 13)
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']


,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


We create a `season` column: a season starts on 1 July, so a match in September 2023
belongs to season 2023 (2023/24) and a match in March 2024 also belongs to season 2023.

In [60]:
# Convert dates and assign each match to a season starting on 1 July
app["date"] = pd.to_datetime(app["date"])
app["season"] = app["date"].dt.year - (app["date"].dt.month < 7)
print("Period:", app["date"].min(), "->", app["date"].max())

# Overall size of the dataset
print("Players:", app["player_id"].nunique())
print("Matches:", app["game_id"].nunique())
print("Competitions:", app["competition_id"].nunique())

# Missing values per column (we expect almost none)
print(app.isna().sum())

Period: 2012-07-03 00:00:00 -> 2026-06-28 00:00:00
Players: 29531
Matches: 73426
Competitions: 48
appearance_id             0
game_id                   0
player_id                 0
player_club_id            0
player_current_club_id    0
date                      0
player_name               2
competition_id            0
yellow_cards              0
red_cards                 0
goals                     0
assists                   0
minutes_played            0
season                    0
dtype: int64


**What we expect to see:** roughly 6,500 to 9,000 players per season, with a dip in
2019/20 (season interrupted by Covid) and a rise in 2025/26 (wider coverage).
The Saka example should show the typical breakout signal: very few minutes,
then a sudden jump in playing time.

In [61]:
# Number of distinct players per season
print(app.groupby("season")["player_id"].nunique())

# Concrete example: minutes, goals and assists of Bukayo Saka per season (exemple)
saka = app[app["player_name"] == "Bukayo Saka"]
print(saka.groupby("season")[["minutes_played", "goals", "assists"]].sum())

season
2012    6635
2013    6797
2014    6458
2015    6769
2016    6782
2017    6642
2018    6842
2019    6597
2020    7608
2021    6008
2022    7098
2023    7150
2024    7852
2025    9051
Name: player_id, dtype: int64
        minutes_played  goals  assists
season                                
2018               123      0        0
2019              2257      3        9
2020              3745      8       11
2021              3084     11        7
2022              3744     15       11
2023              3865     20       14
2024              2500     12       13
2025              3279     11       11


## 2. `player_valuations` — the target

One row per market-value update of a player. This file defines what a *breakout* is,
so it is the most important one to understand.

**What we want to learn:** the period covered, how often a player is re-valued
(if it is several times a year, "the value on 1 July" is well defined),
and the typical size of market values.

In [62]:
# Load the file and look at its size and columns
val = pd.read_csv(config.DATA_RAW / "player_valuations.csv")
print(val.shape)
print(val.columns.tolist())
val.head()

(656301, 6)
['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


In [63]:
# Period covered and number of players with at least one valuation
val["date"] = pd.to_datetime(val["date"])
print("Period:", val["date"].min(), "->", val["date"].max())
print("Players with a valuation:", val["player_id"].nunique())

# How many updates per player and per calendar year?
val["year"] = val["date"].dt.year
print(val.groupby(["player_id", "year"]).size().describe())

Period: 2000-01-20 00:00:00 -> 2026-06-12 00:00:00
Players with a valuation: 41528
count    330838.000000
mean          1.983753
std           0.822471
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          10.000000
dtype: float64


**What we expect to see:** a very skewed distribution — most players are worth
less than a few million euros, a small number are worth far more.
The share of values above 5 million tells us whether that threshold is reasonable.

In [64]:
# Distribution of market values, in millions of euros
print((val["market_value_in_eur"] / 1e6).describe())
print("Share of values >= 5m EUR:", (val["market_value_in_eur"] >= 5e6).mean())

count    656301.000000
mean          2.290093
std           6.721809
min           0.000000
25%           0.200000
50%           0.500000
75%           1.500000
max         200.000000
Name: market_value_in_eur, dtype: float64
Share of values >= 5m EUR: 0.10385021506900036


### First test of the target

For each player and season we keep the **last value known before 1 July**,
then compare it with the value one season later.

A player-season is a **breakout** if the next value is at least **twice** the current one
**and** at least **5 million euros**.

**What we expect to see:** a few percent of breakouts per season.
If there are fewer than about 100 breakouts in total, the thresholds are too strict.

*Caveat:* `shift(-1)` takes the next row of the same player. If a player has no value
for a season, this is not exactly "one year later". This is fixed when building the panel;
here we only want a first order of magnitude.

In [65]:
# Value on 1 July of each season = last valuation before that date
val["season"] = val["date"].dt.year - (val["date"].dt.month < 7)
july = (val.sort_values("date")
           .groupby(["player_id", "season"])["market_value_in_eur"].last()
           .reset_index(name="value"))

# Value one season later, for the same player
july["next_value"] = july.groupby("player_id")["value"].shift(-1)

# Breakout: value at least doubles AND reaches at least 5m EUR
july["breakout"] = (july["next_value"] >= 2 * july["value"]) & (july["next_value"] >= 5e6)

# Number and share of breakouts per season
print(july.groupby("season")["breakout"].agg(["sum", "mean"]))

        sum      mean
season               
1999      0  0.000000
2001      0  0.000000
2003      0  0.000000
2004     83  0.040967
2005     86  0.045575
2006     84  0.037567
2007    123  0.027784
2008    128  0.021560
2009    118  0.016788
2010    133  0.015510
2011    174  0.017594
2012    100  0.008470
2013    188  0.015440
2014    195  0.013279
2015    274  0.016727
2016    473  0.025447
2017    525  0.027732
2018    198  0.009451
2019    468  0.021790
2020    323  0.013048
2021    447  0.017525
2022    494  0.018575
2023    486  0.021793
2024    561  0.031080
2025      0  0.000000


## 3. `players` — age and position

One row per player, with profile information.

**What we want to learn:** whether birth date and position are available for almost everyone
(needed for the age feature and to keep attackers and midfielders), and how players
are spread across positions.

In [66]:
# Load the file and look at its size and columns
players = pd.read_csv(config.DATA_RAW / "players.csv")
print(players.shape)
print(players.columns.tolist())

(50149, 26)
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']


In [67]:
# Share of missing values for the columns we may use
cols = ["player_id", "name", "date_of_birth", "position", "sub_position",
        "country_of_citizenship", "foot", "height_in_cm"]
print(players[cols].isna().mean())

# How players are spread across positions
print(players["position"].value_counts())
print(players["sub_position"].value_counts().head(15))

player_id                 0.000000
name                      0.000000
date_of_birth             0.000977
position                  0.000000
sub_position              0.011685
country_of_citizenship    0.005364
foot                      0.117510
height_in_cm              0.087041
dtype: float64
position
Defender      15961
Midfield      14272
Attack        13502
Goalkeeper     5828
Missing         586
Name: count, dtype: int64
sub_position
Centre-Back           8766
Centre-Forward        6749
Goalkeeper            5828
Central Midfield      5498
Defensive Midfield    4123
Right-Back            3700
Attacking Midfield    3520
Left-Back             3495
Left Winger           3297
Right Winger          3170
Left Midfield          566
Right Midfield         565
Second Striker         286
Name: count, dtype: int64


## 4. `clubs` — club information

One row per club.

**Important:** columns such as `total_market_value` or `squad_size` describe the club
**today**, not in a given past season. Using them as features would leak information from
the future. Club strength will instead be computed **season by season** from the values of its players.

In [68]:
# Load the file and look at its size and columns
clubs = pd.read_csv(config.DATA_RAW / "clubs.csv")
print(clubs.shape)
print(clubs.columns.tolist())
clubs.head()

(796, 17)
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']


,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris FC,FR1,NaN,27,27.2,14,51.9,7,Stade Jean Bouin,19904,€-25.00m,Stéphane Gilli,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,32,26.6,9,28.1,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,US Lecce,IT1,NaN,39,23.8,36,92.3,10,Ettore Giardiniero,31559,+€3.35m,Eusebio Di Francesco,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


In [69]:
# Number of clubs per domestic league
print(clubs["domestic_competition_id"].value_counts())

domestic_competition_id
TR1     43
IT1     39
UKR1    39
GB1     37
FR1     36
PO1     35
RU1     34
ES1     33
ARG1    32
L1      31
MLS1    30
NL1     29
GR1     29
BE1     28
BRA1    24
JAP1    23
DK1     21
SA1     21
PL1     21
COL1    20
SE1     19
NO1     19
RO1     19
MEX1    18
SER1    18
SC1     17
TS1     17
RSK1    14
AUS1    13
A1      13
C1      13
KR1     11
Name: count, dtype: int64


## 5. `competitions` — leagues versus cups

One row per competition.

**What we want to learn:** which competitions are domestic leagues and which are cups or
European competitions. This lets us count league minutes separately and define a league level.

**What we expect to see:** 14 domestic leagues (Italy, Spain, England, France, Turkey,
Germany, Netherlands, Portugal, Belgium, Russia, Greece, Scotland, Ukraine, Denmark).

In [70]:
# Load the file and show the main descriptive columns
comp = pd.read_csv(config.DATA_RAW / "competitions.csv")
print(comp.shape)
print(comp.columns.tolist())
comp[["competition_id", "name", "type", "sub_type", "country_name"]]

(65, 11)
['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']


,competition_id,name,type,sub_type,country_name
0,A1,bundesliga,domestic_league,first_tier,Austria
1,AFAC,afc-asian-cup,national_team_competition,afc_asian_cup,NaN
2,AFCN,africa-cup-of-nations,national_team_competition,africa_cup_of_nations,NaN
3,ARG1,torneo-apertura,domestic_league,first_tier,Argentina
4,AUS1,a-league-men,domestic_league,first_tier,Australia
...,...,...,...,...,...
60,TS1,chance-liga,domestic_league,first_tier,Czech Republic
61,UCOL,uefa-conference-league,other,uefa_conference_league,NaN
62,UKR1,premier-liga,domestic_league,first_tier,Ukraine
63,UKRP,ukrainian-cup,domestic_cup,domestic_cup,Ukraine


In [71]:
# Keep only the domestic leagues
print(comp[comp["type"] == "domestic_league"][["competition_id", "name", "country_name"]])

   competition_id                           name    country_name
0              A1                     bundesliga         Austria
3            ARG1                torneo-apertura       Argentina
4            AUS1                   a-league-men       Australia
5             BE1             jupiler-pro-league         Belgium
8            BRA1  campeonato-brasileiro-serie-a          Brazil
9              C1                   super-league     Switzerland
17            DK1                      superliga         Denmark
23            ES1                         laliga           Spain
27            FR1                        ligue-1          France
29            GB1                 premier-league         England
31            GR1                 super-league-1          Greece
33            IT1                        serie-a           Italy
34           JAP1                      j1-league           Japan
35            KR1                 supersport-hnl         Croatia
36             L1        

## 6. Cross-check — real size of the sample

A player-season is only usable if we have **both** his minutes (from `appearances`)
and his market value on 1 July (from `player_valuations`).

**What we expect to see:** a large majority of player-seasons with minutes also have a value.
The remaining ones will be dropped.

In [72]:
# Player-seasons with playing time
perf = app.groupby(["player_id", "season"])["minutes_played"].sum().reset_index()

# Attach the 1 July value and check how many player-seasons have one
merged = perf.merge(july, on=["player_id", "season"], how="left")
print("Player-seasons with minutes:", len(perf))
print("Share that also has a value:", merged["value"].notna().mean())

Player-seasons with minutes: 98289
Share that also has a value: 0.9517239975989175


## 7. `game_lineups` — starters versus substitutes (optional)

One row per player per match, saying whether he started, came off the bench, or stayed on the bench.

**Why it could matter:** `appearances` only gives minutes. A player who starts every match and a player
who comes on for 30 minutes look different to a coach. The share of matches started could be a stronger
signal of a coming breakout than minutes alone.

In [73]:
# game_lineups: starters vs. substitutes (only a few columns, the file is large)
lineups = pd.read_csv(config.DATA_RAW / "game_lineups.csv", nrows=5)
print(lineups.columns.tolist())      # check the real column names first

['game_lineups_id', 'date', 'game_id', 'player_id', 'club_id', 'player_name', 'type', 'position', 'number', 'team_captain']


## 7. League coverage over time

The `competitions` file lists 31 domestic leagues, but `appearances` may not cover all of them
in every season. The jump to about 9,000 players in 2025/26 suggests that some leagues were added recently.

**Why it matters:** if a league only appears in the most recent seasons, its players have no history
before that, and the sample would not be comparable across seasons.

**What this cell does:** counts the number of appearances per domestic league and per season.
A row with zeros in early seasons and positive numbers later means the league was added during the period.

In [74]:
# Domestic leagues according to the competitions file
leagues = comp.loc[comp["type"] == "domestic_league", "competition_id"]

# Number of appearances per league (rows) and per season (columns)
coverage = (app[app["competition_id"].isin(leagues)]
            .groupby(["competition_id", "season"]).size()
            .unstack(fill_value=0))

# Number of seasons in which each league has at least one match
coverage["n_seasons"] = (coverage > 0).sum(axis=1)
print(coverage.sort_values("n_seasons"))

season           2012   2013   2014   2015   2016   2017   2018  2019   2020  \
competition_id                                                                 
BE1              6571   6588   6597   6617   6629   6618   6592  6368   8711   
DK1              5432   5460   4945   5484   5046   5066   5032  5092   3995   
ES1             10553  10586   9451  10521  10559  10554  10551  9157  12904   
FR1             10433  10474  10297  10559  10496  10515  10523  7710  11456   
GB1             10385  10451   9981  10468  10474  10447  10454  8639  12295   
GR1              6678   8377   7630   6658   6694   6662   6686  5069   5473   
IT1             10521  10581  10106  10593  10607  10587  10549  7934  14644   
L1               8461   8466   8426   8483   8509   8508   8496  8749   9253   
NL1              8286   8347   8387   8361   8440   8375   8396  6378   8600   
PO1              6644   6662   7583   8286   8535   8544   8445  7266  10834   
RU1              6576   6593   6205   65

## 8. Breakouts among attackers and midfielders

The main analysis keeps only attackers and midfielders, because goals and assists measure their
performance well. This cell checks how many breakouts remain once defenders and goalkeepers are removed.

**What we expect to see:** a breakout rate of a few percent per season, and still several hundred
breakouts per season, enough to train the models.

In [75]:
# Attach the position of each player to the player-season values
pos = players[["player_id", "position"]]
july_pos = july.merge(pos, on="player_id", how="left")

# Keep attackers and midfielders only
outfield = july_pos[july_pos["position"].isin(["Attack", "Midfield"])]

# Number and share of breakouts per season
print(outfield.groupby("season")["breakout"].agg(["sum", "mean"]))

        sum      mean
season               
1999      0  0.000000
2003      0  0.000000
2004     51  0.049419
2005     59  0.058128
2006     52  0.044180
2007     81  0.034512
2008     83  0.025736
2009     91  0.023766
2010     80  0.017116
2011    126  0.023213
2012     67  0.010322
2013    130  0.019345
2014    137  0.016953
2015    172  0.018962
2016    296  0.028716
2017    324  0.030708
2018    138  0.011798
2019    300  0.024783
2020    207  0.014821
2021    288  0.020077
2022    334  0.022279
2023    297  0.023726
2024    383  0.038125
2025      0  0.000000


To see if our criteria make sense, we'll highlight a few players who are among these many breakout players

In [76]:
# Sanity check: 20 random breakouts, to see if we recognise the names
breakouts = july[july["breakout"] & july["season"].between(2012, 2024)]
breakouts = breakouts.merge(players[["player_id", "name", "position"]], on="player_id", how="left")
# breakouts = breakouts[breakouts["position"].isin(["Attack", "Midfield"])]   # uncomment to keep attackers and midfielders only

sample = breakouts.sample(20, random_state=42).sort_values("season")

# Readable labels: "2017/18" = season of the starting value, the jump happens during the next one
fmt = lambda s: s.astype(str) + "/" + (s + 1).astype(str).str[-2:]
sample["jump_during"] = fmt(sample["season"] + 1)
sample["season"] = fmt(sample["season"])
sample["value_m"] = (sample["value"] / 1e6).round(1)            # in millions of euros
sample["next_value_m"] = (sample["next_value"] / 1e6).round(1)

sample[["name", "position", "season", "jump_during", "value_m", "next_value_m"]]


,name,position,season,jump_during,value_m,next_value_m
354,Marc Bartra,Defender,2012/13,2013/14,6.0,12.0
199,Aatif Chahechouhe,Midfield,2012/13,2013/14,2.5,5.0
227,Matt Phillips,Attack,2012/13,2013/14,2.5,5.0
1101,Matheus Dória,Defender,2012/13,2013/14,2.0,8.0
1258,Simone Scuffet,Goalkeeper,2013/14,2014/15,2.5,6.0
33,Daniel Carriço,Defender,2013/14,2014/15,3.0,8.0
296,Marvin Plattenhardt,Defender,2015/16,2016/17,3.5,8.0
896,Shoya Nakajima,Midfield,2016/17,2017/18,0.3,8.0
1803,Martin Ødegaard,Midfield,2017/18,2018/19,4.0,15.0
3182,Alexis Mac Allister,Midfield,2018/19,2019/20,5.0,11.0


## Takeaways



- **Seasons usable:** appearances cover 2012/13 to 2025/26. Season 2012 is only used as the "previous season"
  for momentum features, and season 2025 has no target (the data stops in June 2026).
  Player-seasons used for modelling: **2013 to 2024**.
- **Target:** with the thresholds "value at least doubles and reaches at least 5m EUR", there are about
  **3,000 breakouts** among attackers and midfielders between 2013 and 2024, i.e. **1.2% to 3.8% per season**.
  The thresholds are kept.
- **Class imbalance:** breakouts are rare (about 2% on average), so accuracy is not a useful metric;
  precision, recall and F1 will be used.
- **Base rate varies over time:** the share of breakouts moves with the market (e.g. 1.2% in 2018, 3.8% in 2024).
  The test season (2024) has more breakouts than the training seasons on average.
- **Positions kept:** attackers and midfielders (about 27,800 players), because goals and assists measure
  their performance well. Birth date and position are available for almost all players.
- **Competitions:** only **14 domestic leagues** have matches in `appearances`, and all 14 are covered
  in every season, so the sample is comparable over time. The other leagues listed in `competitions` have no matches.
- **Covid season:** 2019/20 was interrupted and finished in July–August 2020, and five substitutions were
  allowed from 2020. Raw minutes are therefore not comparable for 2019 and 2020; a relative measure
  (share of the club's minutes played) will be used.
- **Leakage risks:** columns describing the current situation (`player_current_club_id`, current market value,
  contract, `clubs.total_market_value`) are not used. Club strength is computed season by season.
- **Sample size:** 98,289 player-seasons have minutes, and 95% of them also have a 1 July value.